# LeetCode 3855 — Sum of K-Digit Numbers in a Range bằng Python

- Link: https://leetcode.com/problems/sum-of-k-digit-numbers-in-a-range/
- Độ khó: **Hard**
- Chủ đề: Toán tổ hợp (đếm đóng góp theo vị trí), cấp số nhân, lũy thừa nhanh modulo, nghịch đảo modulo, lũy thừa ma trận

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def sumOfNumbers(self, l: int, r: int, k: int) -> int:
```

## 1. Mục tiêu học

1. Đếm **đóng góp theo từng vị trí chữ số** thay vì liệt kê số: mỗi chữ số ở mỗi vị trí xuất hiện bao nhiêu lần?
2. Rút ra công thức đóng: `tổng = (tổng các chữ số cho phép) · n^{k-1} · (10^k − 1)/9`, với `n = r − l + 1`.
3. Tính `a^b mod p` bằng **lũy thừa nhanh** (`pow(a, b, p)`) và chia cho 9 bằng **nghịch đảo modulo**.
4. Viết lời giải bằng **quy hoạch động theo vị trí** rồi tăng tốc bằng **lũy thừa ma trận 2×2** (không cần nghịch đảo — dùng được với mọi modulo).
5. Kiểm thử với liệt kê trực tiếp.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho `l`, `r`, `k`. Xét mọi xâu gồm **đúng `k` chữ số**, mỗi chữ số được chọn **độc lập** trong đoạn `[l, r]` (nếu `0` thuộc đoạn thì được phép có số 0 đứng đầu). Coi mỗi xâu là một số nguyên; tính **tổng** của tất cả các số đó, modulo `10^9 + 7`.

**Ví dụ 1:** `l = 1, r = 2, k = 2` → `11 + 12 + 21 + 22 = 66`.

**Ví dụ 2:** `l = 0, r = 1, k = 3` → `0 + 1 + 10 + 11 + 100 + 101 + 110 + 111 = 444`.

**Ví dụ 3:** `l = r = 5, k = 10` → `5555555555 mod (10^9+7) = 555555520`.

**Ràng buộc:** `0 ≤ l ≤ r ≤ 9`, `1 ≤ k ≤ 10^9`.

In [1]:
from itertools import product
import random, time

MOD = 10**9 + 7
examples = [(1, 2, 2, 66), (0, 1, 3, 444), (5, 5, 10, 555555520)]
for l, r, k, exp in examples:
    print(l, r, k, "-> kỳ vọng", exp)

1 2 2 -> kỳ vọng 66
0 1 3 -> kỳ vọng 444
5 5 10 -> kỳ vọng 555555520


## 3. Kiến thức nền

### 3.1. Lũy thừa nhanh `pow(a, b, m)`

`pow(a, b, m)` tính `a^b mod m` trong `O(log b)` phép nhân bằng bình phương liên tiếp. Ví dụ `10^{10^9} mod p` tính tức thì.

In [2]:
def fast_pow(a, b, m):
    res = 1
    a %= m
    while b:
        if b & 1:
            res = res * a % m
        a = a * a % m
        b >>= 1
    return res

print(fast_pow(10, 10**9, MOD), pow(10, 10**9, MOD))

142857001 142857001


### 3.2. Nghịch đảo modulo

`MOD` là số nguyên tố ⇒ `x^{-1} ≡ x^{MOD−2}`. "Chia cho 9" theo modulo = nhân với `inv9 = pow(9, MOD − 2, MOD)`. Điều kiện: số bị chia thật sự chia hết cho 9 trong số nguyên (ở đây `10^k − 1` luôn chia hết cho 9), còn khi làm việc theo modulo thì nhân nghịch đảo luôn cho đúng kết quả của phép chia nguyên.

In [3]:
inv9 = pow(9, MOD - 2, MOD)
k = 7
print((10**k - 1) // 9, (pow(10, k, MOD) - 1) * inv9 % MOD)

1111111 1111111


### 3.3. Cấp số nhân `1 + 10 + 10^2 + … + 10^{k−1}`

Tổng bằng `(10^k − 1) / 9` = số `11…1` (`k` chữ số `1`, gọi là *repunit*).

In [4]:
for k in range(1, 6):
    print(k, sum(10**i for i in range(k)), (10**k - 1) // 9)

1 1 1
2 11 11
3 111 111
4 1111 1111
5 11111 11111


### 3.4. Tổng một dãy số nguyên liên tiếp

`l + (l+1) + … + r = (l + r)(r − l + 1)/2`. Tích `(l + r)(r − l + 1)` luôn chẵn nên chia 2 nguyên được.

In [5]:
for l, r in [(1, 2), (0, 9), (3, 7), (5, 5)]:
    print((l, r), sum(range(l, r + 1)), (l + r) * (r - l + 1) // 2)

(1, 2) 3 3
(0, 9) 45 45
(3, 7) 25 25
(5, 5) 5 5


### 3.5. Đổi thứ tự lấy tổng (kỹ thuật "đếm theo đóng góp")

Thay vì `Σ_{mỗi đối tượng} (Σ_{mỗi thành phần} giá trị)`, ta viết `Σ_{mỗi thành phần} (giá trị · số đối tượng chứa thành phần đó)`. Ví dụ: tổng độ dài mọi đoạn con của mảng độ dài `n` = `Σ_i (số đoạn chứa vị trí i) = Σ_i (i + 1)(n − i)`. Bài này dùng đúng tư duy đó với "thành phần" là cặp (vị trí, chữ số).

In [6]:
n = 6
direct = sum(r - l + 1 for l in range(n) for r in range(l, n))
by_contribution = sum((i + 1) * (n - i) for i in range(n))
print("tổng độ dài mọi đoạn con:", direct, "| theo đóng góp:", by_contribution)

tổng độ dài mọi đoạn con: 56 | theo đóng góp: 56


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1 (`l = 1, r = 2, k = 2`): các số `11, 12, 21, 22`.

- **Hàng đơn vị**: chữ số `1` xuất hiện 2 lần, `2` xuất hiện 2 lần ⇒ đóng góp `(1 + 2) · 2 · 1 = 6`.
- **Hàng chục**: tương tự ⇒ `(1 + 2) · 2 · 10 = 60`.
- Tổng `66` ✔.

Tổng quát: có `n = r − l + 1` lựa chọn cho mỗi vị trí. Cố định vị trí `i` (giá trị `10^i`) và chữ số `x` ở đó: `k − 1` vị trí còn lại tự do ⇒ `n^{k−1}` số. Vậy đóng góp của vị trí `i` là `10^i · n^{k−1} · Σ_{x=l..r} x`. Cộng qua mọi `i`:

`Tổng = (Σ_{x=l..r} x) · n^{k−1} · (1 + 10 + … + 10^{k−1})`.

Mọi vị trí **đối xứng** — chỉ khác nhau hệ số `10^i`.

In [7]:
# Kiểm chứng "mỗi (vị trí, chữ số) xuất hiện n^(k-1) lần" bằng đếm trực tiếp
from collections import Counter
l, r, k = 2, 4, 3
freq = Counter()
for d in product(range(l, r + 1), repeat=k):
    for pos, x in enumerate(reversed(d)):
        freq[(pos, x)] += 1
print("tần suất (vị trí, chữ số):", dict(freq))
print("mọi giá trị đều bằng n^(k-1) =", (r - l + 1) ** (k - 1), ":", set(freq.values()))

tần suất (vị trí, chữ số): {(0, 2): 9, (1, 2): 9, (2, 2): 9, (0, 3): 9, (0, 4): 9, (1, 3): 9, (1, 4): 9, (2, 3): 9, (2, 4): 9}
mọi giá trị đều bằng n^(k-1) = 9 : {9}


Bảng tần suất cho thấy tính **đối xứng hoàn toàn**: không có vị trí hay chữ số nào được "ưu tiên". Đây là chìa khóa để thay tổng hàm mũ bằng một tích ba thừa số. Lưu ý: tính đối xứng này **mất đi** nếu có ràng buộc giữa các vị trí (ví dụ "không có số 0 đầu" hay "chữ số tăng dần") — khi đó cần DP chữ số.

## 5. Cách 1 — Brute Force: liệt kê mọi xâu

`itertools.product(range(l, r+1), repeat=k)` cho `n^k` xâu. Chỉ dùng khi `n^k` nhỏ.

In [8]:
def brute(l, r, k):
    total = 0
    for digits in product(range(l, r + 1), repeat=k):
        total += int("".join(map(str, digits)))
    return total % MOD

for l, r, k, exp in examples[:2]:
    print(brute(l, r, k), "kỳ vọng", exp)
print("ví dụ 3 (1 số):", brute(5, 5, 10))

66 kỳ vọng 66
444 kỳ vọng 444
ví dụ 3 (1 số): 555555520


**Nút thắt:** `n^k` với `k = 10^9`. Phải đếm đóng góp, không liệt kê.

## 6. Ý tưởng thuật toán / bước đột phá

**Đột phá:** tách tổng theo **vị trí** và theo **chữ số**: mỗi cặp (vị trí `i`, chữ số `x`) xuất hiện đúng `n^{k−1}` lần. Do đó

`answer = S · n^{k−1} · R_k (mod p)`, với `S = (l + r)·n / 2`, `R_k = (10^k − 1)/9`.

**Ba phép toán khó với `k = 10^9`:**

1. `n^{k−1}` ⇒ lũy thừa nhanh.
2. `10^k` ⇒ lũy thừa nhanh.
3. `/ 9` ⇒ nhân nghịch đảo modulo.

**Cách khác (không cần nghịch đảo):** xây số từ trái sang phải. Gọi `C_i` = số xâu độ dài `i`, `T_i` = tổng giá trị của chúng. Thêm một chữ số vào cuối: `C_{i+1} = n·C_i`, `T_{i+1} = 10·n·T_i + S·C_i`. Đây là hệ tuyến tính ⇒ lũy thừa ma trận `2×2`.

## 7. Các cách tiếp theo

- **Cách 2:** DP theo vị trí `O(k)` (hệ `C`, `T`).
- **Cách 3:** công thức đóng + lũy thừa nhanh + nghịch đảo — `O(log k)` (lời giải nộp chính).
- **Cách 4:** lũy thừa ma trận `2×2` của hệ ở Cách 2 — `O(log k)`, không cần nghịch đảo.

### 7.1. Cách 2 — DP theo vị trí `O(k)`

In [9]:
def dp_linear(l, r, k):
    n = r - l + 1
    S = (l + r) * n // 2
    C, T = 1, 0                      # xâu rỗng: 1 xâu, tổng 0
    for _ in range(k):
        C, T = n * C % MOD, (10 * n * T + S * C) % MOD
    return T

for l, r, k, exp in examples:
    print(dp_linear(l, r, k), "kỳ vọng", exp)

66 kỳ vọng 66
444 kỳ vọng 444
555555520 kỳ vọng 555555520


**Giải thích công thức `T_{i+1}`:** mỗi xâu cũ `v` sinh ra `n` xâu mới `10·v + x` (`x ∈ [l, r]`). Tổng = `Σ_v Σ_x (10v + x) = 10·n·T_i + C_i·S`.

**Độ phức tạp:** `O(k)` — `10^9` vòng lặp Python là quá chậm (~ vài phút).

### 7.2. Cách 3 — Công thức đóng (lời giải chính)

**Giải thích từng đoạn:**

1. `n = r − l + 1`, `S = (l + r)·n // 2` (số nhỏ, tính nguyên rồi lấy modulo).
2. `part1 = pow(n, k − 1, MOD)`.
3. `part2 = (pow(10, k, MOD) − 1) · inv9 mod MOD`.
4. Nhân ba thừa số.

In [10]:
def closed_form(l, r, k):
    n = r - l + 1
    S = (l + r) * n // 2 % MOD
    part1 = pow(n, k - 1, MOD)
    repunit = (pow(10, k, MOD) - 1) * pow(9, MOD - 2, MOD) % MOD
    return S * part1 % MOD * repunit % MOD

for l, r, k, exp in examples:
    print(closed_form(l, r, k), "kỳ vọng", exp)

66 kỳ vọng 66
444 kỳ vọng 444
555555520 kỳ vọng 555555520


**Độ phức tạp Cách 3:** `O(log k)` thời gian, `O(1)` bộ nhớ.

### 7.3. Cách 4 — Lũy thừa ma trận

Viết `[T, C]` dạng vector cột: `[T_{i+1}; C_{i+1}] = M · [T_i; C_i]` với `M = [[10n, S], [0, n]]`. Khi đó `[T_k; C_k] = M^k · [0; 1]`, tức `T_k = M^k[0][1]`.

In [11]:
def mat_mul(A, B):
    return [[(A[0][0] * B[0][0] + A[0][1] * B[1][0]) % MOD, (A[0][0] * B[0][1] + A[0][1] * B[1][1]) % MOD],
            [(A[1][0] * B[0][0] + A[1][1] * B[1][0]) % MOD, (A[1][0] * B[0][1] + A[1][1] * B[1][1]) % MOD]]

def mat_pow(M, e):
    R = [[1, 0], [0, 1]]
    while e:
        if e & 1:
            R = mat_mul(R, M)
        M = mat_mul(M, M)
        e >>= 1
    return R

def matrix_solution(l, r, k):
    n = r - l + 1
    S = (l + r) * n // 2
    M = [[10 * n % MOD, S % MOD], [0, n % MOD]]
    return mat_pow(M, k)[0][1]

for l, r, k, exp in examples:
    print(matrix_solution(l, r, k), "kỳ vọng", exp)

66 kỳ vọng 66
444 kỳ vọng 444
555555520 kỳ vọng 555555520


In [12]:
class SolutionMatrix:
    def sumOfNumbers(self, l: int, r: int, k: int) -> int:
        return matrix_solution(l, r, k)

print([SolutionMatrix().sumOfNumbers(l, r, k) for l, r, k, _ in examples])

[66, 444, 555555520]


**Độ phức tạp Cách 4:** `O(log k)` phép nhân ma trận `2×2` (8 phép nhân mỗi lần). Ưu điểm: **không cần** `MOD` nguyên tố.

## 8. TRACE từng bước

Trace Cách 2 với `l = 0, r = 1, k = 3` (`n = 2`, `S = 1`):

| i | C_i (số xâu) | T_i (tổng) | giải thích |
|---|---|---|---|
| 0 | 1 | 0 | xâu rỗng |
| 1 | 2 | 10·2·0 + 1·1 = 1 | {0, 1} |
| 2 | 4 | 10·2·1 + 1·2 = 22 | {00, 01, 10, 11} = 0+1+10+11 |
| 3 | 8 | 10·2·22 + 1·4 = 444 | ✔ |

In [13]:
def trace_dp(l, r, k):
    n = r - l + 1; S = (l + r) * n // 2
    C, T = 1, 0
    print(f"n={n}, S={S}")
    for i in range(1, k + 1):
        C, T = n * C, 10 * n * T + S * C
        print(f"i={i}: C={C}, T={T}")
    return T

trace_dp(0, 1, 3)
print("liệt kê:", sum(int("".join(map(str, d))) for d in product(range(2), repeat=3)))

n=2, S=1
i=1: C=2, T=1
i=2: C=4, T=22
i=3: C=8, T=444
liệt kê: 444


In [14]:
# Trace công thức đóng cho ví dụ 1 và ví dụ 3
for l, r, k in [(1, 2, 2), (5, 5, 10)]:
    n = r - l + 1; S = (l + r) * n // 2
    print(f"l={l} r={r} k={k}: S={S}, n^(k-1)={n ** (k - 1)}, repunit={(10 ** k - 1) // 9}, tích={S * n ** (k - 1) * ((10 ** k - 1) // 9)}, mod={closed_form(l, r, k)}")

l=1 r=2 k=2: S=3, n^(k-1)=2, repunit=11, tích=66, mod=66
l=5 r=5 k=10: S=5, n^(k-1)=1, repunit=1111111111, tích=5555555555, mod=555555520


Trace **lũy thừa ma trận** (Cách 4) với `l = 0, r = 1, k = 3`: `M = [[20, 1], [0, 2]]`. Lũy thừa nhanh duyệt các bit của `k = 3 = 0b11`: nhân `R` với `M`, bình phương `M`, nhân tiếp. Kết quả `M^3 = [[8000, 444], [0, 8]]` — phần tử `[0][1] = 444` chính là đáp án, phần tử `[1][1] = 8 = n^k` là số xâu.

In [15]:
def trace_mat_pow(M, e):
    R = [[1, 0], [0, 1]]; step = 0
    while e:
        if e & 1:
            R = mat_mul(R, M); print(f"bước {step}: bit 1 -> R = {R}")
        else:
            print(f"bước {step}: bit 0 -> giữ R")
        M = mat_mul(M, M); e >>= 1; step += 1
        print(f"          M bình phương -> {M}")
    return R

R = trace_mat_pow([[20, 1], [0, 2]], 3)
print("T_3 =", R[0][1], "| C_3 =", R[1][1])

bước 0: bit 1 -> R = [[20, 1], [0, 2]]
          M bình phương -> [[400, 22], [0, 4]]
bước 1: bit 1 -> R = [[8000, 444], [0, 8]]
          M bình phương -> [[160000, 8888], [0, 16]]
T_3 = 444 | C_3 = 8


## 9. Code nộp LeetCode

In [16]:
class Solution:
    def sumOfNumbers(self, l: int, r: int, k: int) -> int:
        MOD = 10**9 + 7
        n = r - l + 1
        S = (l + r) * n // 2                      # tổng các chữ số được phép
        count_rest = pow(n, k - 1, MOD)           # số cách chọn k-1 vị trí còn lại
        repunit = (pow(10, k, MOD) - 1) * pow(9, MOD - 2, MOD) % MOD   # 11...1 (k chữ số)
        return S % MOD * count_rest % MOD * repunit % MOD

sol = Solution()
for l, r, k, exp in examples:
    got = sol.sumOfNumbers(l, r, k)
    print((l, r, k), "->", got, "✔" if got == exp else "✘")

(1, 2, 2) -> 66 ✔
(0, 1, 3) -> 444 ✔
(5, 5, 10) -> 555555520 ✔


## 10. Vì sao thuật toán đúng

1. **Song ánh xâu ↔ bộ chữ số:** mỗi số được xét là một bộ `(d_{k−1}, …, d_0)` với `d_i ∈ [l, r]`, giá trị `Σ d_i 10^i`. Có `n^k` bộ.
2. **Đổi thứ tự lấy tổng:** `Σ_{bộ} Σ_i d_i 10^i = Σ_i 10^i Σ_{bộ} d_i`. Với `i` cố định, mỗi giá trị `x` của `d_i` đi kèm `n^{k−1}` cách chọn phần còn lại ⇒ `Σ_{bộ} d_i = n^{k−1}·S`.
3. **Cấp số nhân:** `Σ_{i<k} 10^i = (10^k − 1)/9` (nguyên). Theo modulo nguyên tố `p ≠ 3`, phép chia cho 9 tương đương nhân nghịch đảo.
4. **Cách 2/4:** quy nạp theo độ dài: hệ thức `T_{i+1} = 10nT_i + SC_i` đúng như giải thích ở 7.1; lũy thừa ma trận chỉ là áp dụng hệ thức `k` lần.

**Vì sao `MOD = 10^9 + 7` cho phép chia 9?** Nghịch đảo của `a` modulo `p` tồn tại khi và chỉ khi `gcd(a, p) = 1`. `p = 10^9 + 7` là số nguyên tố và không chia hết cho 3, nên `9` có nghịch đảo. Nếu đề dùng modulo `999999999 = 9 · 111111111` thì **không** nghịch đảo được — khi đó dùng Cách 4 (ma trận) hoặc tính repunit bằng chia để trị (Bài tập 3).

In [17]:
from math import gcd
for p in (10**9 + 7, 998244353, 999999999):
    print(p, "gcd(9, p) =", gcd(9, p), "-> có nghịch đảo" if gcd(9, p) == 1 else "-> KHÔNG có nghịch đảo")

1000000007 gcd(9, p) = 1 -> có nghịch đảo
998244353 gcd(9, p) = 1 -> có nghịch đảo
999999999 gcd(9, p) = 9 -> KHÔNG có nghịch đảo


**Kiểm chứng bằng số học thuần (không modulo).** Với `k` nhỏ, công thức nguyên `S · n^{k−1} · (10^k − 1)/9` phải trùng **chính xác** với tổng liệt kê — không chỉ trùng theo modulo. Đây là cách kiểm tra mạnh hơn, vì sai sót kiểu "đúng theo modulo một cách tình cờ" bị loại trừ.

In [18]:
def exact_formula(l, r, k):
    n = r - l + 1
    return (l + r) * n // 2 * n ** (k - 1) * ((10**k - 1) // 9)

ok = all(exact_formula(l, r, k) == sum(int("".join(map(str, d))) for d in product(range(l, r + 1), repeat=k))
         for l in range(10) for r in range(l, 10) for k in range(1, 4))
print("công thức nguyên khớp liệt kê với mọi (l, r) và k <= 3:", ok)

công thức nguyên khớp liệt kê với mọi (l, r) và k <= 3: True


## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê | `O(n^k · k)` | `O(k)` |
| 2. DP theo vị trí | `O(k)` | `O(1)` |
| 3. Công thức đóng | `O(log k)` | `O(1)` |
| 4. Lũy thừa ma trận | `O(8 log k)` | `O(1)` |

## 12. Bộ test

In [19]:
def run_all(l, r, k):
    res = [closed_form(l, r, k), matrix_solution(l, r, k), Solution().sumOfNumbers(l, r, k)]
    if k <= 10**5:
        res.append(dp_linear(l, r, k))
    if k <= 12 and (r - l + 1) ** k <= 2 * 10**5:
        res.append(brute(l, r, k))
    return res

for l, r, k, exp in examples:
    assert all(v == exp for v in run_all(l, r, k))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [20]:
# 12.2 ca biên (tự dựng)
cases = [
    (0, 0, 1, 0), (0, 0, 10**9, 0),      # chỉ có số 0
    (9, 9, 1, 9), (0, 9, 1, 45),         # một chữ số
    (0, 9, 2, sum(range(100))),          # mọi số 00..99
    (1, 9, 10**9, None),                 # k cực lớn
]
for l, r, k, exp in cases:
    res = run_all(l, r, k)
    assert len(set(res)) == 1, (l, r, k, res)
    if exp is not None:
        assert res[0] == exp % MOD
    print((l, r, k), "->", res[0])

(0, 0, 1) -> 0
(0, 0, 1000000000) -> 0
(9, 9, 1) -> 9
(0, 9, 1) -> 45
(0, 9, 2) -> 4950
(1, 9, 1000000000) -> 293270712


In [21]:
# 12.3 phản ví dụ: quên rằng mỗi vị trí còn lại có n lựa chọn (nhân n^k thay vì n^(k-1))
def wrong_power(l, r, k):
    n = r - l + 1; S = (l + r) * n // 2
    return S * pow(n, k, MOD) % MOD * ((pow(10, k, MOD) - 1) * pow(9, MOD - 2, MOD)) % MOD

print("sai:", wrong_power(1, 2, 2), "| đúng:", Solution().sumOfNumbers(1, 2, 2))

sai: 132 | đúng: 66


In [22]:
# 12.4 đối chiếu ngẫu nhiên
rng = random.Random(3855)
for _ in range(300):
    l = rng.randint(0, 9); r = rng.randint(l, 9); k = rng.randint(1, 6)
    res = run_all(l, r, k)
    assert len(set(res)) == 1, (l, r, k, res)
for _ in range(200):
    l = rng.randint(0, 9); r = rng.randint(l, 9); k = rng.randint(1, 10**9)
    res = run_all(l, r, k)
    assert len(set(res)) == 1, (l, r, k, res)
print("500 test ngẫu nhiên: OK")

500 test ngẫu nhiên: OK


In [23]:
# 12.5 tốc độ với k = 10^9
t0 = time.time()
for _ in range(10000):
    Solution().sumOfNumbers(0, 9, 10**9)
print("10000 lần gọi:", round(time.time() - t0, 3), "s")

10000 lần gọi: 0.037 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chia cho 9 bằng `//` sau khi đã lấy modulo

`(pow(10, k, MOD) − 1) // 9` **sai** vì sau modulo số đó không còn chia hết cho 9 (hoặc chia ra giá trị khác).

In [24]:
k = 20
wrong = (pow(10, k, MOD) - 1) // 9 % MOD
right = (pow(10, k, MOD) - 1) * pow(9, MOD - 2, MOD) % MOD
print("sai:", wrong, "| đúng:", right, "| kiểm tra trực tiếp:", (10**k - 1) // 9 % MOD)

sai: 544 | đúng: 333333880 | kiểm tra trực tiếp: 333333880


### Lỗi 2 — Dùng `n^{k}` thay cho `n^{k−1}`

Đã thấy ở 12.3: khi cố định chữ số tại một vị trí, chỉ còn `k − 1` vị trí tự do.

### Lỗi 3 — Tính `10**k` bằng số nguyên Python khi `k = 10^9`

Python hỗ trợ số lớn, nhưng `10**(10**9)` có một tỉ chữ số — tốn hàng GB bộ nhớ và rất chậm. **Sửa:** `pow(10, k, MOD)`.

In [25]:
t0 = time.time(); x = 10 ** 200000; t1 = time.time()
print("10^200000 cần", x.bit_length(), "bit (~", x.bit_length() // 8 // 1024, "KB), mất", round(t1 - t0, 4), "s; 10^(10^9) sẽ lớn gấp 5000 lần (~400 MB)")
# Python còn chặn việc in số quá 4300 chữ số (str(x) báo lỗi) — một dấu hiệu nữa là không nên làm vậy

10^200000 cần 664386 bit (~ 81 KB), mất 0.0122 s; 10^(10^9) sẽ lớn gấp 5000 lần (~400 MB)


### Lỗi 4 — Bỏ qua số 0 đầu (chỉ đếm số "thật sự" `k` chữ số)

Đề nói: nếu `0` thuộc đoạn thì **được phép** có số 0 đầu (ví dụ 2 tính cả `000`, `001`, …). Nếu bỏ các xâu có 0 đầu sẽ ra sai.

In [26]:
def wrong_no_leading_zero(l, r, k):
    return sum(int("".join(map(str, d))) for d in product(range(l, r + 1), repeat=k) if d[0] != 0) % MOD

print("sai:", wrong_no_leading_zero(0, 1, 3), "| đúng:", Solution().sumOfNumbers(0, 1, 3))

sai: 422 | đúng: 444


### Lỗi 5 — Tràn số khi chuyển sang C++/Java

Python không tràn, nhưng trong C++/Java tích ba số mỗi số `< 10^9 + 7` vượt `2^63`. Phải lấy modulo **sau từng phép nhân**: `a * b % MOD * c % MOD`, không phải `a * b * c % MOD`. Đoạn dưới mô phỏng giới hạn 64-bit để thấy giá trị trung gian vượt ngưỡng.

In [27]:
a = b = c = MOD - 1
LIM = 2**63 - 1
print("a*b vượt 2^63?", a * b > LIM, "| a*b*c vượt 2^63?", a * b * c > LIM)
print("a*b % MOD * c vượt 2^63?", (a * b % MOD) * c > LIM)

a*b vượt 2^63? False | a*b*c vượt 2^63? True
a*b % MOD * c vượt 2^63? False


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Tổng của tất cả các số/xâu tạo được"** ⇒ đếm **đóng góp** của mỗi (vị trí, giá trị) — tính đối xứng thường cho công thức đóng.
- **Số mũ lớn tới `10^9`** ⇒ lũy thừa nhanh; **chia theo modulo** ⇒ nghịch đảo (khi modulo nguyên tố) hoặc lũy thừa ma trận / cấp số nhân chia để trị (khi không).
- Hệ thức tuyến tính `T_{i+1} = a·T_i + b·C_i` ⇒ lũy thừa ma trận.
- Liên quan: *Sum of Digits of All Numbers*, *Count Good Numbers* (LC 1922 — lũy thừa nhanh).

**Mẹo kiểm tra công thức nhanh:** với `l = r = d` chỉ có một số `dd…d` (`k` chữ số) ⇒ đáp án phải là `d · repunit(k)`. Với `l = 0, r = 9` mọi số từ `0` tới `10^k − 1` xuất hiện đúng một lần ⇒ đáp án `= (10^k − 1)·10^k / 2`. Hai "điểm neo" này giúp phát hiện sai sót trong công thức trước khi nộp.

In [28]:
for d, k in [(5, 10), (7, 3), (1, 50)]:
    assert Solution().sumOfNumbers(d, d, k) == d * ((10**k - 1) // 9) % MOD
for k in (1, 2, 5, 30):
    assert Solution().sumOfNumbers(0, 9, k) == (10**k - 1) * 10**k // 2 % MOD
print("Hai điểm neo đều khớp.")

Hai điểm neo đều khớp.


## 15. Biến thể của bài

1. **Mỗi vị trí có đoạn riêng** `[l_i, r_i]`: tổng = `Σ_i 10^i · S_i · Π_{j≠i} n_j` (cần `k` nhỏ).
2. **Cấm số 0 đầu**: vị trí cao nhất dùng tập `[max(l,1), r]`.
3. **Cơ số `b` khác 10**: thay `10` bằng `b`, repunit `(b^k − 1)/(b − 1)`.
4. **Tập chữ số bất kỳ** (không liên tiếp): `S` = tổng tập, `n` = kích thước tập.

In [29]:
# Biến thể 2: không cho phép số 0 đầu (vị trí cao nhất dùng [max(l,1), r])
def no_leading_zero(l, r, k):
    n = r - l + 1; S = (l + r) * n // 2
    lo = max(l, 1)
    if lo > r: return 0
    n0 = r - lo + 1; S0 = (lo + r) * n0 // 2
    if k == 1: return S0 % MOD
    inv9 = pow(9, MOD - 2, MOD)
    low = S * pow(n, k - 2, MOD) % MOD * n0 % MOD * ((pow(10, k - 1, MOD) - 1) * inv9 % MOD) % MOD
    top = S0 * pow(n, k - 1, MOD) % MOD * pow(10, k - 1, MOD) % MOD
    return (low + top) % MOD

for l, r, k in [(0, 1, 3), (0, 2, 2), (0, 9, 3), (3, 5, 4)]:
    assert no_leading_zero(l, r, k) == wrong_no_leading_zero(l, r, k)
print("Biến thể 2 khớp liệt kê.")

Biến thể 2 khớp liệt kê.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê | Dựng mọi xâu | `O(n^k k)` | `O(k)` | Kiểm thử |
| 2. DP vị trí | `C`, `T` theo độ dài | `O(k)` | `O(1)` | `k ≤ 10^6` |
| 3. Công thức đóng | `S·n^{k−1}·repunit` | `O(log k)` | `O(1)` | **Lời giải chuẩn** |
| 4. Lũy thừa ma trận | `M^k` | `O(log k)` | `O(1)` | Modulo không nguyên tố |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `sumOfNumbers(2, 3, 1)` = ?

**Bài 2 (tính tay).** `sumOfNumbers(1, 3, 2)` — tính bằng công thức.

**Bài 3 (điền TODO).** Viết `repunit_mod(k)` = `(10^k − 1)/9 mod p` **không dùng nghịch đảo** (chia để trị: `R_{2m} = R_m·(10^m + 1)`).

**Bài 4 (tự viết lại).** Viết lời giải cho **cơ số `b`** bất kỳ (`2 ≤ b ≤ 10`, chữ số trong `[l, r] ⊆ [0, b−1]`).

**Bài 5 (mini challenge).** Mỗi vị trí `i` (tính từ hàng đơn vị) có đoạn riêng `[l_i, r_i]`, `k ≤ 10^5`. Tính tổng mọi số (gợi ý: tích tiền tố/hậu tố của `n_j`).

In [30]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def repunit_mod(k):
    # TODO: R(0) = 0, R(1) = 1, R(2m) = R(m) * (10^m + 1), R(m+1) = 10*R(m) + 1
    pass

In [31]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def sum_per_position(ranges):
    # TODO: ranges[i] = (l_i, r_i); tổng = Σ_i 10^i * S_i * Π_{j != i} n_j
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `k = 1`. (2) Các số `2, 3`. (3) `5`.

**Bài 2.** (1) `n = 3`, `S = 6`. (2) `S · n^{1} · 11 = 6 · 3 · 11`. (3) `198`.

**Bài 3.** (1) Đệ quy theo bit của `k`. (2) Mỗi bước cần `10^m mod p`. (3) Kiểm tra với `(10^k − 1)//9 % p` khi `k` nhỏ.

**Bài 4.** (1) Thay `10` bằng `b`. (2) Repunit `(b^k − 1)/(b − 1)`. (3) Cẩn thận `b − 1` có nghịch đảo modulo `p`.

**Bài 5.** (1) `P = Π n_j`. (2) `Π_{j≠i} n_j = P / n_i` — nhưng dùng tiền tố/hậu tố để tránh chia. (3) Cộng `10^i · S_i · prefix[i] · suffix[i+1]`.

## 19. Đáp án tham khảo cho bài tập

In [32]:
# Bài 1, 2
assert Solution().sumOfNumbers(2, 3, 1) == 5
assert Solution().sumOfNumbers(1, 3, 2) == 198 == brute(1, 3, 2)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [33]:
# Bài 3
def repunit_mod_sol(k):
    if k == 0:
        return 0
    if k % 2 == 0:
        h = repunit_mod_sol(k // 2)
        return h * (pow(10, k // 2, MOD) + 1) % MOD
    return (10 * repunit_mod_sol(k - 1) + 1) % MOD

for k in [1, 2, 3, 10, 37, 1000]:
    assert repunit_mod_sol(k) == (10**k - 1) // 9 % MOD
assert repunit_mod_sol(10**9) == (pow(10, 10**9, MOD) - 1) * pow(9, MOD - 2, MOD) % MOD
print("Bài 3 OK")

Bài 3 OK


In [34]:
# Bài 4: cơ số b
def sum_base(l, r, k, b):
    n = r - l + 1; S = (l + r) * n // 2
    rep = (pow(b, k, MOD) - 1) * pow(b - 1, MOD - 2, MOD) % MOD
    return S * pow(n, k - 1, MOD) % MOD * rep % MOD

def brute_base(l, r, k, b):
    tot = 0
    for d in product(range(l, r + 1), repeat=k):
        v = 0
        for x in d: v = v * b + x
        tot += v
    return tot % MOD

rng = random.Random(4)
for _ in range(200):
    b = rng.randint(2, 10); l = rng.randint(0, b - 1); r = rng.randint(l, b - 1); k = rng.randint(1, 5)
    assert sum_base(l, r, k, b) == brute_base(l, r, k, b)
print("Bài 4 OK")

Bài 4 OK


In [35]:
# Bài 5: mỗi vị trí một đoạn riêng
def sum_per_position_sol(ranges):
    k = len(ranges)
    ns = [r - l + 1 for l, r in ranges]
    pre = [1] * (k + 1); suf = [1] * (k + 1)
    for i in range(k): pre[i + 1] = pre[i] * ns[i] % MOD
    for i in range(k - 1, -1, -1): suf[i] = suf[i + 1] * ns[i] % MOD
    tot = 0
    for i, (l, r) in enumerate(ranges):
        S = (l + r) * ns[i] // 2
        tot = (tot + pow(10, i, MOD) * S % MOD * pre[i] % MOD * suf[i + 1]) % MOD
    return tot

def brute_pos(ranges):
    tot = 0
    for d in product(*[range(l, r + 1) for l, r in ranges]):
        tot += sum(x * 10**i for i, x in enumerate(d))
    return tot % MOD

rng = random.Random(5)
for _ in range(200):
    rs = []
    for _ in range(rng.randint(1, 4)):
        l = rng.randint(0, 9); rs.append((l, rng.randint(l, 9)))
    assert sum_per_position_sol(rs) == brute_pos(rs)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
n = r - l + 1
S = (l + r) * n // 2
ans = S * pow(n, k - 1, MOD) * (pow(10, k, MOD) - 1) * pow(9, MOD - 2, MOD) % MOD
```

**Checklist:**

- [ ] Số mũ của `n` là `k − 1` (không phải `k`)?
- [ ] Chia 9 bằng **nghịch đảo**, không dùng `//` sau khi lấy modulo?
- [ ] Không tính `10**k` bằng số nguyên lớn?
- [ ] Tính cả các xâu có số 0 đầu khi `l = 0`?
- [ ] Lấy modulo ở mỗi phép nhân?